# Hybrid Funnel Entity Resolution Pipeline

**Architecture**: Dense E5 Blocking → RapidFuzz Feature Engineering → LightGBM GBDT Reranker → Hard Veto Shield

**Metric**: $F_{0.5}$ (Precision-weighted)

**OOM Safeguards**: float16 disk-streamed embeddings, chunked FAISS, merge-based feature joins, numpy feature arrays

In [ ]:
# CELL 1: Environment & Hardware
import os, sys, gc, re, time, pickle
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
import faiss
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import fbeta_score, classification_report

try:
    from rapidfuzz import distance, fuzz
    HAS_RAPIDFUZZ = True
except ImportError:
    HAS_RAPIDFUZZ = False

DEVICE = "cpu"
if torch.cuda.is_available():
    DEVICE = "cuda"
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    try:
        import torch_directml
        DEVICE = torch_directml.device()
        print(f"GPU: DirectML ({DEVICE})")
    except ImportError:
        torch.set_num_threads(max(1, os.cpu_count() or 8))
        print("CPU mode")

print(f"PyTorch {torch.__version__} | RapidFuzz: {HAS_RAPIDFUZZ}")

In [ ]:
# CELL 2: Paths & Data Loading
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.exists("../data") else os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, "data", "processed")
OUT_DIR = os.path.join(BASE_DIR, "data", "output", "pipeline1")
os.makedirs(OUT_DIR, exist_ok=True)

FILES = {
    "train_s1": "train_source1_clean.parquet",
    "train_s2": "train_source2_clean.parquet",
    "train_s3": "train_source3_clean.parquet",
    "train_gt": "train_ground_truth.tsv",
    "test_s1":  "test_source1_clean.parquet",
    "test_s2":  "test_source2_clean.parquet",
    "test_s3":  "test_source3_clean.parquet",
}

def load(key):
    f = FILES[key]
    # Ground truth: check multiple locations
    if key == "train_gt":
        for p in [
            os.path.join(DATA_DIR, f),
            os.path.join(BASE_DIR, "Dataset", "student_resource", "dataset", "train", "train_ground_truth.tsv"),
        ]:
            if os.path.exists(p):
                print(f"  GT: {p}")
                return pd.read_csv(p, sep="\t")
    path = os.path.join(DATA_DIR, f)
    print(f"  {key}: {len(pd.read_parquet(path)):,} rows" if f.endswith(".parquet") else f"  {key}: {path}")
    return pd.read_parquet(path) if f.endswith(".parquet") else pd.read_csv(path, sep="\t")

print(f"Data: {DATA_DIR}\nOutput: {OUT_DIR}")

In [ ]:
# CELL 3: Stage 1 — Dense Blocking (E5 + float16 memmap + chunked FAISS)
E5_MODEL = "intfloat/multilingual-e5-small"
BATCH_SIZE = 256
TOP_K = 5

def get_id_col(df):
    for c in ["source1_entity_id", "entity_id"]:
        if c in df.columns:
            return c
    return df.columns[0]

def get_text(df):
    n = df["name_clean"].fillna("").astype(str) if "name_clean" in df.columns else df.iloc[:, 1].fillna("").astype(str)
    a = df["addr_clean"].fillna("").astype(str) if "addr_clean" in df.columns else ""
    return (n + " " + a).str.strip()

def dense_blocking(query_df, target_df, top_k=TOP_K, cache_key="data"):
    """Float16 memory-mapped encoding + chunked FAISS search. Zero RAM overflow risk."""
    q_path = os.path.join(DATA_DIR, f"{cache_key}_q_{len(query_df)}.npy")
    t_path = os.path.join(DATA_DIR, f"{cache_key}_t_{len(target_df)}.npy")

    if os.path.exists(q_path) and os.path.exists(t_path):
        print("  Loading cached embeddings...")
        q_emb = np.load(q_path, mmap_mode="r")
        t_emb = np.load(t_path, mmap_mode="r")
    else:
        print(f"  Encoding with {E5_MODEL} on {DEVICE}...")
        tokenizer = AutoTokenizer.from_pretrained(E5_MODEL)
        try:
            model = AutoModel.from_pretrained(E5_MODEL).to(DEVICE)
        except Exception:
            model = AutoModel.from_pretrained(E5_MODEL).to("cpu")
        model.eval()

        def encode(texts, prefix, out_path):
            n, dim = len(texts), 384
            mm = np.lib.format.open_memmap(out_path, mode="w+", dtype="float16", shape=(n, dim))
            for i in tqdm(range(0, n, BATCH_SIZE), desc=f"  {prefix[:5]}"):
                batch = [prefix + t for t in texts[i:i + BATCH_SIZE]]
                try:
                    inp = tokenizer(batch, max_length=128, padding=True, truncation=True, return_tensors="pt").to(DEVICE)
                    with torch.no_grad():
                        out = model(**inp)
                except Exception:
                    inp = tokenizer(batch, max_length=128, padding=True, truncation=True, return_tensors="pt").to("cpu")
                    with torch.no_grad():
                        out = model.to("cpu").float()(**inp)
                mask = inp["attention_mask"].unsqueeze(-1).expand(out.last_hidden_state.size()).float()
                pooled = (out.last_hidden_state.to("cpu") * mask.to("cpu")).sum(1) / torch.clamp(mask.to("cpu").sum(1), min=1e-9)
                mm[i:i + len(batch)] = F.normalize(pooled, p=2, dim=1).numpy().astype(np.float16)
            mm.flush()
            return mm

        q_emb = encode(get_text(query_df).tolist(), "query: ", q_path)
        t_emb = encode(get_text(target_df).tolist(), "passage: ", t_path)
        del model, tokenizer
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        gc.collect()

    # Chunked FAISS search (500k index chunks, 50k query chunks)
    print(f"  FAISS search (top-{top_k})...")
    index = faiss.IndexFlatIP(q_emb.shape[1])
    for c in range(0, len(t_emb), 500_000):
        index.add(np.asarray(t_emb[c:c + 500_000], dtype=np.float32))

    D_all, I_all = [], []
    for c in range(0, len(q_emb), 50_000):
        d, i = index.search(np.asarray(q_emb[c:c + 50_000], dtype=np.float32), top_k)
        D_all.append(d); I_all.append(i)
    D = np.vstack(D_all); I = np.vstack(I_all)
    del index; gc.collect()

    q_ids = query_df[get_id_col(query_df)].values
    t_ids = target_df[get_id_col(target_df)].values
    df = pd.DataFrame({
        "source1_entity_id": np.repeat(q_ids, top_k),
        "candidate_entity_id": t_ids[I.ravel()],
        "dense_score": D.ravel().astype(np.float32)
    })
    print(f"  ✓ {len(df):,} candidate pairs")
    return df

In [ ]:
# CELL 4: Stage 2 — Feature Engineering (merge-based, numpy arrays — no dict OOM)
FEATURE_COLS = [
    "dense_score",
    "name_jaro", "name_ratio", "name_token_sort", "name_token_set",
    "addr_jaro", "addr_ratio", "addr_token_sort", "addr_token_set",
    "country_match", "postal_match", "house_num_overlap"
]

def _str_sim(s1, s2):
    a, b = str(s1 or "").strip().lower(), str(s2 or "").strip().lower()
    if not a or not b:
        return 0., 0., 0., 0.
    if HAS_RAPIDFUZZ:
        return (distance.JaroWinkler.similarity(a, b), fuzz.ratio(a, b) / 100.,
                fuzz.token_sort_ratio(a, b) / 100., fuzz.token_set_ratio(a, b) / 100.)
    from difflib import SequenceMatcher
    r = SequenceMatcher(None, a, b).ratio()
    w1, w2 = set(a.split()), set(b.split())
    t = len(w1 & w2) / max(len(w1 | w2), 1)
    return r, r, t, t

def _house_nums(text):
    return set(re.findall(r'\b\d+\b', str(text))) if text and not pd.isna(text) else set()

def build_features(cands, query_df, target_df):
    """Merge-based feature extraction: joins only needed columns, stores in numpy arrays."""
    q_id = get_id_col(query_df)
    t_id = get_id_col(target_df)
    
    # Select only needed columns — avoids materializing 10M-row dicts
    q = query_df[[q_id, "name_clean", "addr_clean", "country", "postal_code"]].drop_duplicates(subset=[q_id])
    q = q.rename(columns={q_id: "source1_entity_id", "name_clean": "n1", "addr_clean": "a1", "country": "c1", "postal_code": "p1"})
    
    t = target_df[[t_id, "name_clean", "addr_clean", "country", "postal_code"]].drop_duplicates(subset=[t_id])
    t = t.rename(columns={t_id: "candidate_entity_id", "name_clean": "n2", "addr_clean": "a2", "country": "c2", "postal_code": "p2"})
    
    # Merge candidate pairs with entity fields (RAM-efficient: only candidate rows materialized)
    m = cands.merge(q, on="source1_entity_id", how="left").merge(t, on="candidate_entity_id", how="left")
    del q, t; gc.collect()
    
    n = len(m)
    print(f"  Computing features for {n:,} pairs...")
    
    # Extract arrays (avoid iterrows overhead entirely)
    n1 = m["n1"].fillna("").values
    n2 = m["n2"].fillna("").values
    a1 = m["a1"].fillna("").values
    a2 = m["a2"].fillna("").values
    c1 = m["c1"].fillna("").astype(str).str.strip().str.upper().values
    c2 = m["c2"].fillna("").astype(str).str.strip().str.upper().values
    p1 = m["p1"].fillna("").astype(str).str.strip().values
    p2 = m["p2"].fillna("").astype(str).str.strip().values
    
    # Pre-allocate numpy arrays (12 features × 4 bytes × N ≈ 500 MB for 11M pairs — vs ~18 GB for list-of-dicts)
    nj = np.zeros(n, dtype=np.float32)
    nr = np.zeros(n, dtype=np.float32)
    nts = np.zeros(n, dtype=np.float32)
    ntset = np.zeros(n, dtype=np.float32)
    aj = np.zeros(n, dtype=np.float32)
    ar = np.zeros(n, dtype=np.float32)
    ats = np.zeros(n, dtype=np.float32)
    atset = np.zeros(n, dtype=np.float32)
    cm = np.full(n, -1.0, dtype=np.float32)
    pm = np.full(n, -1.0, dtype=np.float32)
    hm = np.full(n, -1.0, dtype=np.float32)
    
    for i in tqdm(range(n), desc="  Features"):
        nj[i], nr[i], nts[i], ntset[i] = _str_sim(n1[i], n2[i])
        aj[i], ar[i], ats[i], atset[i] = _str_sim(a1[i], a2[i])
        
        if c1[i] and c2[i]:
            cm[i] = 1.0 if c1[i] == c2[i] else 0.0
        if p1[i] and p2[i] and len(p1[i]) >= 3:
            pm[i] = 1.0 if p1[i] == p2[i] else 0.0
        h1, h2 = _house_nums(a1[i]), _house_nums(a2[i])
        if h1 and h2:
            hm[i] = 1.0 if len(h1 & h2) > 0 else 0.0
    
    feat_df = pd.DataFrame({
        "source1_entity_id": m["source1_entity_id"].values,
        "candidate_entity_id": m["candidate_entity_id"].values,
        "dense_score": m["dense_score"].values,
        "name_jaro": nj, "name_ratio": nr, "name_token_sort": nts, "name_token_set": ntset,
        "addr_jaro": aj, "addr_ratio": ar, "addr_token_sort": ats, "addr_token_set": atset,
        "country_match": cm, "postal_match": pm, "house_num_overlap": hm,
    })
    del m; gc.collect()
    print(f"  ✓ Features extracted")
    return feat_df

In [ ]:
# CELL 5: Stage 3 — LightGBM Training + F_0.5 Calibration
print("Loading train data...")
train_s1 = load("train_s1")
train_s2 = load("train_s2")
train_s3 = load("train_s3")
train_gt = load("train_gt")

train_s23 = pd.concat([train_s2, train_s3], ignore_index=True)
del train_s2, train_s3; gc.collect()

print("\nBlocking...")
train_cands = dense_blocking(train_s1, train_s23, cache_key="train")

# Vectorized GT labeling via set lookup
print("Labeling...")
gt_pairs = train_gt.dropna(subset=["matched_entity_ids"]).copy()
gt_pairs["matched_entity_ids"] = gt_pairs["matched_entity_ids"].astype(str)
gt_exp = gt_pairs.assign(cand=gt_pairs["matched_entity_ids"].str.split(",")).explode("cand")
gt_set = set(zip(gt_exp["source1_entity_id"].astype(str).str.strip(), gt_exp["cand"].str.strip()))
labels = np.array([
    1 if (str(s).strip(), str(c).strip()) in gt_set else 0
    for s, c in zip(train_cands["source1_entity_id"], train_cands["candidate_entity_id"])
], dtype=np.int8)
print(f"  Pairs: {len(labels):,} | Positives: {labels.sum():,}")
del gt_exp, gt_pairs, gt_set; gc.collect()

print("\nFeature extraction...")
train_feats = build_features(train_cands, train_s1, train_s23)
del train_cands; gc.collect()

print("\nTraining LightGBM...")
X = train_feats[FEATURE_COLS]
y = labels
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

clf = lgb.LGBMClassifier(
    n_estimators=300, learning_rate=0.05, num_leaves=31,
    max_depth=6, class_weight="balanced", random_state=42, n_jobs=-1
)
clf.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(30, verbose=False)])

va_probs = clf.predict_proba(X_va)[:, 1]
best_thresh, best_f05 = 0.5, 0.0
for th in np.arange(0.30, 0.98, 0.02):
    s = fbeta_score(y_va, (va_probs >= th).astype(int), beta=0.5, zero_division=0)
    if s > best_f05:
        best_f05, best_thresh = s, th

print(f"\n{'='*50}")
print(f"F_0.5: {best_f05:.4f} @ threshold {best_thresh:.2f}")
print(classification_report(y_va, (va_probs >= best_thresh).astype(int), target_names=["NO_MATCH", "MATCH"], digits=4))

del X, y, X_tr, X_va, y_tr, y_va, train_feats, labels; gc.collect()

In [ ]:
# CELL 6: Stage 4 — Test Inference + Hard Veto Shield + Submission
print("Loading test data...")
test_s1 = load("test_s1")
test_s2 = load("test_s2")
test_s3 = load("test_s3")

test_s23 = pd.concat([test_s2, test_s3], ignore_index=True)
del test_s2, test_s3; gc.collect()

print("\nBlocking...")
test_cands = dense_blocking(test_s1, test_s23, cache_key="test")

print("\nFeature extraction...")
test_feats = build_features(test_cands, test_s1, test_s23)
del test_s23; gc.collect()

print("\nScoring + Hard Veto Shield...")
probs = clf.predict_proba(test_feats[FEATURE_COLS])[:, 1]
cm = test_feats["country_match"].values
hm = test_feats["house_num_overlap"].values
nj = test_feats["name_jaro"].values

# Vectorized veto: country conflict | (house mismatch & low name sim) | very low name sim
veto_mask = (cm == 0.0) | ((hm == 0.0) & (nj < 0.85)) | (nj < 0.40)
match_mask = (~veto_mask) & (probs >= best_thresh)

print(f"  Veto rejected: {veto_mask.sum():,} pairs")
print(f"  Final matches: {match_mask.sum():,} pairs")

test_feats["prediction"] = np.where(match_mask, "MATCH", "NO_MATCH")

# Write submission TSVs
print("\nWriting submissions...")
cand_path = os.path.join(OUT_DIR, "candidate_pairs.tsv")
match_path = os.path.join(OUT_DIR, "matching_results.tsv")

cand_grouped = test_cands.groupby("source1_entity_id")["candidate_entity_id"].apply(",".join).reset_index()
cand_grouped.columns = ["source1_entity_id", "candidate_entity_ids"]
cand_grouped.to_csv(cand_path, sep="\t", index=False)

matched = test_feats[test_feats["prediction"] == "MATCH"]
match_grouped = matched.groupby("source1_entity_id")["candidate_entity_id"].apply(",".join).reset_index()
match_grouped.columns = ["source1_entity_id", "matched_entity_ids"]

all_s1 = pd.DataFrame({"source1_entity_id": test_s1[get_id_col(test_s1)].unique()})
final = pd.merge(all_s1, match_grouped, on="source1_entity_id", how="left").fillna("")
final.to_csv(match_path, sep="\t", index=False)

print(f"\n✓ candidate_pairs.tsv  ({len(cand_grouped):,} rows)")
print(f"✓ matching_results.tsv ({len(final):,} rows)")
print(f"\nSaved to: {OUT_DIR}")